<h4>A. Results statistic</h4>

1. List of ERROR products:

In [2]:
import os
from collections import Counter

error_folder = "error_200k"
error_counts = Counter()
total_errors = 0

# 
for filename in os.listdir(error_folder):
    if filename.startswith("errors_batch_") and filename.endswith(".txt"):
        filepath = os.path.join(error_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    total_errors += 1
                    # get "error="
                    if "error=" in line:
                        error_type = line.split("error=")[-1]
                        error_counts[error_type] += 1
                    else:
                        error_counts["Unknown"] += 1

print(f"Numbers of ERROR products: {total_errors}")
print("ERROR Catgeory:")
for err, count in error_counts.items():
    print(f"- {err}: {count}")


Numbers of ERROR products: 9456
ERROR Catgeory:
- HTTP 404: 6638
- HTTP 429: 2810
- 'NoneType' object is not iterable: 8


2. Get 8 products with NoneType status

In [4]:
import os

error_folder = "error_200k"
target_error = "'NoneType' object is not iterable"

for filename in os.listdir(error_folder):
    if filename.startswith("errors_batch_") and filename.endswith(".txt"):
        filepath = os.path.join(error_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            for lineno, line in enumerate(f, start=1):
                if target_error in line:
                    print(f"{filepath} - row {lineno}: {line.strip()}")


error_200k\errors_batch_103.txt - row 20: [ERROR] index=102591, id=7543425, error='NoneType' object is not iterable
error_200k\errors_batch_103.txt - row 46: [ERROR] index=102948, id=7470553, error='NoneType' object is not iterable
error_200k\errors_batch_185.txt - row 37: [ERROR] index=184702, id=9370172, error='NoneType' object is not iterable
error_200k\errors_batch_33.txt - row 1: [ERROR] index=32010, id=9803081, error='NoneType' object is not iterable
error_200k\errors_batch_94.txt - row 2: [ERROR] index=93087, id=7697454, error='NoneType' object is not iterable
error_200k\errors_batch_94.txt - row 12: [ERROR] index=93652, id=7697437, error='NoneType' object is not iterable
error_200k\errors_batch_96.txt - row 31: [ERROR] index=95825, id=7973548, error='NoneType' object is not iterable
error_200k\errors_batch_96.txt - row 37: [ERROR] index=95889, id=7697522, error='NoneType' object is not iterable


3. Get sum of products GET sucessfully

In [18]:
import os
import json
import pandas as pd

result_folder = "result_200k"
total_products = []

for filename in os.listdir(result_folder):
    if filename.startswith("product_batch_") and filename.endswith(".json"):
        filepath = os.path.join(result_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
            products = data.get("data", [])
            for p in products:
                if p and "id" in p:
                    total_products.append(p["id"])

print(f"Total of products in all batches (get successfully): {len(total_products)}")
print(f"Sum of [OK] products and [ERROR] products: {len(total_products) + total_errors}")


Total of products in all batches (get successfully): 190544
Sum of [OK] products and [ERROR] products: 200000


<h4>B. Process ERROR products (excute again)</h4>

1. List of 429 & 404 ERROR products

In [8]:
import os

error_folder = "error_200k"
result_folder = "result_200k"

error_429_ids = []
error_404_ids = []
another_error_ids = []

for filename in os.listdir(error_folder):
    if filename.startswith("errors_batch_") and filename.endswith(".txt"):
        filepath = os.path.join(error_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if "error=HTTP 429" in line:
                    # tách id
                    parts = line.split(",")
                    for p in parts:
                        if p.strip().startswith("id="):
                            pid = p.strip().split("=")[-1]
                            error_429_ids.append(pid)
                elif "error=HTTP 404" in line:
                    # tách id
                    parts = line.split(",")
                    for p in parts:
                        if p.strip().startswith("id="):
                            pid = p.strip().split("=")[-1]
                            error_404_ids.append(pid)
                else:
                    # tách id
                    parts = line.split(",")
                    for p in parts:
                        if p.strip().startswith("id="):
                            pid = p.strip().split("=")[-1]
                            another_error_ids.append(pid)

print(f"Found {len(error_429_ids)} product id with ERROR 429")
print(f"Found {len(error_404_ids)} product id with ERROR 404")
print(f"Found {len(another_error_ids)} product id with other errors")

Found 2810 product id with ERROR 429
Found 6638 product id with ERROR 404
Found 8 product id with other errors


2. Scripts for execution again ERROR product, with 3 times retry for 404 ERROR and 5 times retry for 429 ERROR (2s, 4s, 8s,...)

In [13]:
import aiohttp
import asyncio
import pandas as pd
from bs4 import BeautifulSoup
import json
import logging
import os

# Create folders for results and errors
#os.makedirs("result_20k", exist_ok=True)
#os.makedirs("error_20k", exist_ok=True)

# Configure logging
logging.basicConfig(
    filename="log-20k.txt",
    level=logging.INFO,
    format="%(message)s",
    encoding="utf-8"
)

semaphore = asyncio.Semaphore(10)

async def fetch_with_retry(session, url, row_index, product_id,
                           max_retry_404=3, max_retry_429=5):
    attempt_404 = 0
    attempt_429 = 0

    while True:
        async with semaphore:
            try:
                async with session.get(url) as response:
                    if response.status == 200:
                        try:
                            data = await response.json()
                        except Exception:
                            msg = f"[ERROR] row={row_index}, id={product_id}, error=Can not parse JSON"
                            print(msg)
                            logging.error(msg)
                            return None, msg

                        raw_description = data.get("description", "")
                        soup = BeautifulSoup(raw_description, "html.parser")
                        clean_description = soup.get_text(separator=" ", strip=True)

                        product_info = {
                            "id": data.get("id"),
                            "name": data.get("name"),
                            "url_key": data.get("url_key"),
                            "price": data.get("price"),
                            "description": clean_description,
                            "images": [img.get("base_url") for img in data.get("images", [])]
                        }

                        msg = f"[OK] row={row_index}, id={product_id}"
                        print(msg)
                        logging.info(msg)
                        return product_info, None

                    elif response.status == 404:
                        attempt_404 += 1
                        if attempt_404 >= max_retry_404:
                            msg = f"[ERROR] row={row_index}, id={product_id}, error=HTTP 404 (Not Found after {attempt_404} retries)"
                            print(msg)
                            logging.error(msg)
                            return None, msg
                        await asyncio.sleep(1)  # short delay

                    elif response.status == 429:
                        attempt_429 += 1
                        if attempt_429 >= max_retry_429:
                            msg = f"[ERROR] row={row_index}, id={product_id}, error=HTTP 429 (Too Many Requests after {attempt_429} retries)"
                            print(msg)
                            logging.error(msg)
                            return None, msg
                        await asyncio.sleep(2 ** attempt_429)  # exponential backoff

                    else:
                        msg = f"[ERROR] row={row_index}, id={product_id}, error=HTTP {response.status}"
                        print(msg)
                        logging.error(msg)
                        return None, msg

            except Exception as e:
                msg = f"[ERROR] row={row_index}, id={product_id}, error={str(e)}"
                print(msg)
                logging.error(msg)
                return None, msg

async def process_batch(batch_df, batch_index):
    products = []
    error_msgs = []

    async with aiohttp.ClientSession(headers={"User-Agent": "Mozilla/5.0"}) as session:
        tasks = []
        for row_index, product_id in zip(batch_df.index, batch_df["id"]):
            url = f"https://api.tiki.vn/product-detail/api/v1/products/{product_id}"
            tasks.append(fetch_with_retry(session, url, row_index, product_id))

        results = await asyncio.gather(*tasks)

    for result, err in results:
        if result:
            products.append(result)
        if err:
            error_msgs.append(err)

    # Save batch result json file
    with open(f"get_again_error_products(1)/product_batch_{203+batch_index+1}.json", "w", encoding="utf-8") as f:
        json.dump({"data": products}, f, ensure_ascii=False, indent=4)

    # Save batch error file
    with open(f"get_again_error_products(1)/errors_batch_{203+batch_index+1}.txt", "w", encoding="utf-8") as f:
        for msg in error_msgs:
            f.write(msg + "\n")

    summary = (
        f"=== Batch {batch_index+1} ===\n"
        f"Numbers of products GET successfully: {len(products)}\n"
        f"Numbers of ERROR: {len(error_msgs)}\n"
        f"Saved to get_again_error_products(1)/product_batch_{203+batch_index+1}.json\n"
        f"Saved to get_again_error_products(1)/errors_batch_{203+batch_index+1}.txt\n"
    )
    print(summary)
    logging.info(summary)


3. Run 429 products first (save from 201 to 203 batches file)

In [11]:
async def main():
    #df = pd.read_csv("products-0-200000.csv")
    batch_size = 1000
    num_batches = (len(error_429_ids) + batch_size - 1) // batch_size

    for batch_index in range(num_batches):
        start = batch_index * batch_size
        end = min((batch_index + 1) * batch_size, len(error_429_ids))
        batch_ids = error_429_ids[start:end]
        batch_df = pd.DataFrame({"id": batch_ids})

        print(f"=== Processing batch {batch_index+1}/{num_batches}, including {len(batch_df)} products ===")
        await process_batch(batch_df, batch_index)

if __name__ == "__main__":
    await main()


=== Processing batch 1/3, including 1000 products ===
[OK] row=1, id=176123086
[OK] row=6, id=159270752
[OK] row=2, id=123532946
[OK] row=3, id=162770565
[OK] row=4, id=106377849
[OK] row=7, id=52657325
[OK] row=10, id=208600572
[OK] row=0, id=96995898
[OK] row=12, id=216416910
[OK] row=5, id=75420157
[OK] row=9, id=181153509
[OK] row=13, id=208235759
[OK] row=8, id=90852601
[OK] row=14, id=198458875
[OK] row=11, id=180037101
[OK] row=16, id=174656635
[OK] row=15, id=174656479
[OK] row=21, id=179150654
[OK] row=19, id=171659445
[OK] row=18, id=247806413
[OK] row=17, id=109155766
[OK] row=23, id=152193155
[OK] row=20, id=253079430
[OK] row=22, id=247723081
[OK] row=25, id=109138133
[OK] row=24, id=191651278
[OK] row=26, id=40758338
[OK] row=31, id=235544629
[OK] row=27, id=157826920
[OK] row=30, id=127510733
[OK] row=29, id=130180039
[OK] row=28, id=68305542
[OK] row=33, id=126048964
[OK] row=32, id=247858253
[OK] row=37, id=134318592
[OK] row=35, id=90061430
[OK] row=36, id=146373955
[

4. For 404 ERROR products, run again make sure we have all not found products (save from 204 to 210 batches file)

In [14]:
async def main():
    #df = pd.read_csv("products-0-200000.csv")
    batch_size = 1000
    num_batches = (len(error_404_ids) + batch_size - 1) // batch_size

    for batch_index in range(num_batches):
        start = batch_index * batch_size
        end = min((batch_index + 1) * batch_size, len(error_404_ids))
        batch_ids = error_404_ids[start:end]
        batch_df = pd.DataFrame({"id": batch_ids})

        print(f"=== Processing batch {batch_index+1}/{num_batches}, including {len(batch_df)} products ===")
        await process_batch(batch_df, batch_index)

if __name__ == "__main__":
    await main()


=== Processing batch 1/7, including 1000 products ===
[ERROR] row=9, id=214841295, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=2, id=193513488, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=6, id=154530131, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=7, id=215162868, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=4, id=202719601, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=8, id=139457802, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=3, id=154530205, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=1, id=191463934, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=5, id=252177965, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=10, id=253041020, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=0, id=184319033, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=11, id=212054438, error=HTTP 404 (Not Found after 3 retries)
[ERROR] row=12, id=196562199, error=HTTP 404 (Not Found after 3 retries)
[ERROR]

5. Save to CSV list of [OK] products and [ERROR] products:

In [19]:
import os
import json
import pandas as pd

result_folder = "get_again_error_products"
more_products = []

for filename in os.listdir(result_folder):
    if filename.startswith("product_batch_") and filename.endswith(".json"):
        filepath = os.path.join(result_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
            products = data.get("data", [])
            for p in products:
                if p and "id" in p:
                    more_products.append(p["id"])

print(f"More products: {len(more_products)}")

#combine together
all_products = total_products + more_products

df = pd.DataFrame({"id": all_products})
df.to_csv("all_products.csv", index=False, encoding="utf-8")
print("Saved to all_products.csv")


More products: 2698
Saved to all_products.csv


In [23]:
error_folder = "get_again_error_products"

total_error_404_ids = []

for filename in os.listdir(error_folder):
    if filename.startswith("errors_batch_") and filename.endswith(".txt"):
        filepath = os.path.join(error_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if "error=HTTP 404" in line:
                    # tách id
                    parts = line.split(",")
                    for p in parts:
                        if p.strip().startswith("id="):
                            pid = p.strip().split("=")[-1]
                            total_error_404_ids.append(pid)

print(f"Found {len(total_error_404_ids)} product id with ERROR 404")

df_2 = pd.DataFrame({"id": total_error_404_ids})
df_2.to_csv("error_404_products.csv", index=False, encoding="utf-8")
print("Saved to error_404_products.csv")

Found 6750 product id with ERROR 404
Saved to error_404_products.csv


SUMMARY

In [28]:
print(f"With {len(all_products) + len(total_error_404_ids) + 8} product ids, we get {len(all_products)} products successfully, and {len(total_error_404_ids)} products with ERROR 404, 8 NoneType product objects")

With 200000 product ids, we get 193242 products successfully, and 6750 products with ERROR 404, 8 NoneType product objects
